In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import re
import time
from typing import List, Dict, Any, Tuple, Callable

# [문제 1 정답]
def preprocess_sentences(sentences: List[List[int]], max_l: int) -> Tuple[torch.Tensor, torch.Tensor]:
    batch_ids = []
    batch_masks = []
    for s in sentences:
        ids = [1] + s + [2]
        pad_len = max_l - len(ids)
        mask = [1] * len(ids) + [0] * pad_len
        ids = ids + [0] * pad_len
        batch_ids.append(ids)
        batch_masks.append(mask)
    return torch.tensor(batch_ids, dtype=torch.long), torch.tensor(batch_masks, dtype=torch.long)

# [문제 2 정답]
def split_heads(tensor: torch.Tensor, num_heads: int) -> torch.Tensor:
    batch, seq, hidden = tensor.size()
    head_dim = hidden // num_heads
    return tensor.view(batch, seq, num_heads, head_dim).transpose(1, 2)

# [문제 3 정답]
def append_to_kv_cache(prev_k: torch.Tensor, new_k: torch.Tensor) -> torch.Tensor:
    return torch.cat([prev_k, new_k], dim=-2)

# [문제 4 정답]
def get_probs_with_temp(logits: torch.Tensor, temp: float) -> torch.Tensor:
    return F.softmax(logits / temp, dim=-1)

# [문제 5 정답]
currency_tool_schema = {
    "type": "object",
    "properties": {
        "amount": {"type": "number"},
        "from_cur": {"type": "string"},
        "to_cur": {"type": "string", "default": "USD"}
    },
    "required": ["amount", "from_cur"]
}

# [문제 6 정답]
def parse_model_call(call_str: str) -> Dict[str, Any]:
    match = re.search(r"CALL: (\w+)\((.*)\)", call_str)
    if not match: return {}
    func_name = match.group(1)
    args_raw = match.group(2)
    # 단순화를 위해 쉼표 기준 분리 (실제로는 더 복잡한 파싱 필요)
    args = dict(re.findall(r"(\w+)=(\w+)", args_raw))
    return {"func_name": func_name, "args": args}

# [문제 7 정답]
def update_mha_kv_cache(prev_k: torch.Tensor, new_k_proj: torch.Tensor, num_heads: int) -> torch.Tensor:
    batch, seq_new, hidden = new_k_proj.size()
    head_dim = hidden // num_heads
    # (B, S, H) -> (B, S, nH, D) -> (B, nH, S, D)
    new_k = new_k_proj.view(batch, seq_new, num_heads, head_dim).transpose(1, 2)
    return torch.cat([prev_k, new_k], dim=-2)

# [문제 8 정답 - 킬러 1]
def multi_step_tool_agent(item: str, tools: Dict[str, Callable]) -> str:
    pid = tools["find_product_id"](item)
    if pid is None:
        return "ERROR: PRODUCT_NOT_FOUND"

    discount = tools["get_discount_rate"](pid)
    if discount > 0.5:
        return "ERROR: INVALID_DISCOUNT"

    price = tools["calculate_price"](pid, discount)
    return f"Final Price for {item} is {price}"

# [문제 9 정답]
def sliding_window_batch_kv(cache: torch.Tensor, new_val: torch.Tensor, window_size: int) -> torch.Tensor:
    combined = torch.cat([cache, new_val], dim=-2)
    return combined[:, :, -window_size:, :]

# [문제 10 정답 - 킬러 2]
def call_llm_api_safely(prompt: str, max_tokens: int, api_fn: Callable) -> Dict[str, Any]:
    if len(prompt.split()) > max_tokens:
        return "ERROR: CONTEXT_EXCEEDED"

    attempts = 0
    while attempts <= 2: # 총 3회 시도
        attempts += 1
        status, resp = api_fn(prompt)

        if status == 200:
            return {"status": "success", "response": resp, "attempts": attempts}

        if status == 429:
            if attempts <= 2:
                time.sleep(1)
                continue
            else:
                return {"status": "fail", "reason": "RATE_LIMIT_EXCEEDED"}

        # 500 등 기타 서버 에러
        return {"status": "fail", "reason": "SERVER_ERROR"}

    return {"status": "fail", "reason": "UNKNOWN"}